# Lab: Gaussian Mixture Models, from Densities to a Full Probability Model

**Bluevision-ai Academy**
*Gaussian Mixture Models: Soft Clustering with EM*

In lecture you watched k-means confess two honest failures -- the boundary lie and the wrong shape -- then fixed both by trading a single centre for a full distribution, fitted by maximum likelihood via EM, with k-means falling out the other side as a special case once its freedoms are switched off. This lab makes every one of those claims yourself, in code, on numbers you can check.

Each part runs the same rhythm the lecture did, just in cells instead of clicks: **Problem** (what we have, what we want, why the obvious tool doesn't reach it) -> **Idea** (the reasoning move, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what just happened, and what it means). One idea per cell -- if a cell needs "and" to describe it, it's doing too much.

| Part | What you build | Ties back to |
|---|---|---|
| 1 -- One EM Iteration, by Hand, in Code | Reproduce the slides' five-point E-step/M-step table in NumPy | Slides 12, 15, 17 |
| 2 -- GMM at Scale | Full-covariance GMM vs k-means on elongated data; the spherical reveal | Slides 6, 18, 19 |
| 3 -- A Model, Not a Map | Score outliers, sample new points from the fitted density | Slides 21-22 |
| Challenge | Break it on purpose: the EM singularity, then choose K honestly | Slide 20 |
| Reflection | Five questions, answered in code comments | -- |

**Dataset:** the same five-point 1D running example the slides computed by hand (Part 1), plus a purpose-built elongated three-cluster cloud (Parts 2-3) shaped exactly like slide 6's villain. **Estimated time:** ~2 hours. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab already ships with everything this lab needs. The cell below just makes sure versions are current, turns on interactive widgets, and loads the deck's own palette so these plots read like the ones you just watched.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade scikit-learn plotly ipywidgets scipy

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, fixed

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running on Colab -- local Jupyter with ipywidgets already works

np.random.seed(0)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, GREEN, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#1a7a4a", "#7f8c9a"


## Part 1 -- One EM Iteration, by Hand, in Code

**Problem.** Slides 15 and 17 walked one full EM iteration by hand on five points -- $[1.0, 1.6, 2.8, 4.0, 4.6]$ -- starting from two rough guesses ($\mu_A=1.3$, $\mu_B=4.3$, $\sigma_A=\sigma_B=1.0$) and ending with sharper ones ($\mu_A'=1.63$, $\mu_B'=3.97$, both spreads at $0.71$). Hand arithmetic is easy to get subtly wrong and easy to trust blindly either way -- the only way to know the slide's numbers are the mechanical output of a formula, not a scripted illustration, is to make code produce the exact same digits.

**Ingredients.**

In [2]:
# The five points from the likelihood and E/M-step slides, and the starting guess.
PTS5 = np.array([1.0, 1.6, 2.8, 4.0, 4.6])
mu0, sigma0 = np.array([1.3, 4.3]), np.array([1.0, 1.0])
print("points:", PTS5, "  starting means:", mu0, "  starting spreads:", sigma0)

points: [1.  1.6 2.8 4.  4.6]   starting means: [1.3 4.3]   starting spreads: [1. 1.]


**Idea -- the E-step, as a formula.** For a 1D Gaussian with mean $\mu$ and spread $\sigma$, the density at a point $x$ is
$$p(x \mid \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).$$
Score every point under both starting Gaussians, then turn those two densities into a *responsibility* -- component $k$'s share of explaining point $x$ -- by normalising:
$$\gamma_k(x) = \frac{p(x\mid \mu_k,\sigma_k)}{p(x\mid \mu_A,\sigma_A) + p(x\mid \mu_B,\sigma_B)}.$$
With $\pi_A=\pi_B$ the weights are equal and cancel out of the ratio, so this is exactly the E-step formula from slide 14, just written for one dimension.

In [3]:
def gaussian_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

p_A = gaussian_pdf(PTS5, mu0[0], sigma0[0])
p_B = gaussian_pdf(PTS5, mu0[1], sigma0[1])
gamma_A = p_A / (p_A + p_B)
gamma_B = 1 - gamma_A

for x, pa, pb, ga, gb in zip(PTS5, p_A, p_B, gamma_A, gamma_B):
    print(f"x={x:.1f}   p(x|A)={pa:.4f}   p(x|B)={pb:.4f}   gamma_A={ga:.3f}   gamma_B={gb:.3f}")

assert np.allclose(gamma_A, [0.996, 0.973, 0.500, 0.027, 0.004], atol=1e-3)

x=1.0   p(x|A)=0.3814   p(x|B)=0.0017   gamma_A=0.996   gamma_B=0.004
x=1.6   p(x|A)=0.3814   p(x|B)=0.0104   gamma_A=0.973   gamma_B=0.027
x=2.8   p(x|A)=0.1295   p(x|B)=0.1295   gamma_A=0.500   gamma_B=0.500
x=4.0   p(x|A)=0.0104   p(x|B)=0.3814   gamma_A=0.027   gamma_B=0.973
x=4.6   p(x|A)=0.0017   p(x|B)=0.3814   gamma_A=0.004   gamma_B=0.996


**Payoff.** Every value matches the slide 15 table digit for digit, including the row that matters most: the point at $x=2.8$, sitting exactly midway between the two starting means, comes out at $\gamma_A=\gamma_B=0.500$ -- not because we asked for it, but because the formula produces it automatically whenever a point is equidistant in density from both components. That is villain number one, now computed rather than merely watched: a formula, run honestly, hands the boundary point the answer a hard assignment could never give it.

**Idea -- the M-step, as a formula.** Freeze those responsibilities and refit each Gaussian to its *weighted* share of the data -- the same sample-mean-and-spread fact from slide 12, just with every point counted by how much it belongs, not whether it belongs:
$$N_k=\sum_i \gamma_k(x_i), \qquad \mu_k'=\frac{1}{N_k}\sum_i \gamma_k(x_i)\,x_i, \qquad {\sigma_k'}^2=\frac{1}{N_k}\sum_i \gamma_k(x_i)(x_i-\mu_k')^2, \qquad \pi_k'=\frac{N_k}{n}.$$

In [4]:
N_A, N_B = gamma_A.sum(), gamma_B.sum()
mu_A_new = (gamma_A * PTS5).sum() / N_A
mu_B_new = (gamma_B * PTS5).sum() / N_B
sigma_A_new = np.sqrt((gamma_A * (PTS5 - mu_A_new) ** 2).sum() / N_A)
sigma_B_new = np.sqrt((gamma_B * (PTS5 - mu_B_new) ** 2).sum() / N_B)
pi_A_new, pi_B_new = N_A / 5, N_B / 5

print(f"N_A={N_A:.2f}  N_B={N_B:.2f}")
print(f"mu_A'={mu_A_new:.2f}  mu_B'={mu_B_new:.2f}")
print(f"sigma_A'={sigma_A_new:.2f}  sigma_B'={sigma_B_new:.2f}")
print(f"pi_A'={pi_A_new:.2f}  pi_B'={pi_B_new:.2f}")

assert round(mu_A_new, 2) == 1.63 and round(mu_B_new, 2) == 3.97
assert round(sigma_A_new, 2) == round(sigma_B_new, 2) == 0.71
assert round(pi_A_new, 2) == round(pi_B_new, 2) == 0.5

N_A=2.50  N_B=2.50
mu_A'=1.63  mu_B'=3.97
sigma_A'=0.71  sigma_B'=0.71
pi_A'=0.50  pi_B'=0.50


**Payoff.** Every number matches slide 17 exactly: both means pulled in toward their real cluster ($1.3\to1.63$, $4.3\to3.97$), both spreads tightened ($1.0\to0.71$), and the weights stayed even because the responsibilities really did split 2.5/2.5. The takeaway generalises past these five points: EM is not two different algorithms bolted together -- it is the same maximum-likelihood formula from slide 12 applied twice, once with points counted fully (a single Gaussian) and once with points counted fractionally (a mixture component) -- and alternating those two easy problems is exactly what breaks the chicken-and-egg slide 13 named.

## Part 2 -- GMM at Scale

**Problem.** Slide 6 showed k-means slicing straight through elongated clusters because a single centre can only ever carve round territory. Slides 18-19 then claimed a full-covariance GMM finds that same elongated shape, and that switching its freedoms off one at a time turns it back into k-means. Let's put both claims on measurable ground: build data shaped exactly like slide 6's villain, and see how much of the true structure each algorithm actually recovers.

**Ingredients.**

In [5]:
def make_elongated_cluster(center, angle_deg, n, stretch, base_std, rng):
    # n points, Gaussian in a local frame, stretched then rotated -- an elongated blob.
    local = rng.normal(0, 1, size=(n, 2)) * np.array([base_std * stretch, base_std])
    theta = np.radians(angle_deg)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return local @ R.T + np.array(center)

rng = np.random.default_rng(3)
n_per = 70
c1 = make_elongated_cluster((-3.0, 0.0), 20, n_per, stretch=4.5, base_std=0.4, rng=rng)
c2 = make_elongated_cluster((3.0, 1.2), -30, n_per, stretch=4.5, base_std=0.4, rng=rng)
c3 = make_elongated_cluster((0.0, -3.5), 78, n_per, stretch=4.5, base_std=0.4, rng=rng)
X_elong = np.vstack([c1, c2, c3])
y_true = np.array([0] * n_per + [1] * n_per + [2] * n_per)
print(X_elong.shape, "points across 3 elongated clusters -- the species label (y_true) is hidden from both algorithms below")

(210, 2) points across 3 elongated clusters -- the species label (y_true) is hidden from both algorithms below


**Idea -- fit both, honestly, and score against the truth we hid.** Fit `KMeans` and a full-covariance `GaussianMixture`, both told there are 3 groups, neither shown `y_true`. Score each against the ground truth with the adjusted Rand index (ARI: 1.0 is a perfect match, 0.0 is random agreement). Then look, not just score: colour every point by its GMM responsibility blend -- a point that is 70% component 1 and 30% component 2 gets a dot that is a 70/30 blend of those two colours -- and draw each fitted Gaussian's 2-sigma covariance ellipse alongside k-means' round decision boundary.

In [6]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_elong)
gmm_full = GaussianMixture(n_components=3, covariance_type="full", n_init=5, random_state=0).fit(X_elong)
resp = gmm_full.predict_proba(X_elong)

ari_km = adjusted_rand_score(y_true, km.labels_)
ari_gmm = adjusted_rand_score(y_true, gmm_full.predict(X_elong))
print(f"ARI, k-means:              {ari_km:.3f}")
print(f"ARI, full-covariance GMM:  {ari_gmm:.3f}")

ARI, k-means:              0.791
ARI, full-covariance GMM:  0.986


In [7]:
def cov_ellipse_xy(mean, cov, n_std=2, n_pts=60):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.linspace(0, 2 * np.pi, n_pts)
    circle = np.stack([np.cos(theta), np.sin(theta)])
    ellipse = vecs @ np.diag(n_std * np.sqrt(vals)) @ circle
    return mean[0] + ellipse[0], mean[1] + ellipse[1]

def blend_colors(resp, hexes):
    rgb = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in hexes])
    blended = resp @ rgb
    return [f"rgb({r:.0f},{g:.0f},{b:.0f})" for r, g, b in blended]

comp_colors = [GOLD, BLUE, TEAL]
km_colors = [comp_colors[j] for j in km.labels_]
point_colors = blend_colors(resp, comp_colors)

fig = make_subplots(rows=1, cols=2, subplot_titles=("k-means: round territory", "GMM: shape included"))
fig.add_trace(go.Scatter(x=X_elong[:, 0], y=X_elong[:, 1], mode="markers",
                          marker=dict(color=km_colors, size=5), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=km.cluster_centers_[:, 0], y=km.cluster_centers_[:, 1], mode="markers",
                          marker=dict(symbol="x", size=14, color=RED, line=dict(width=3)),
                          showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=X_elong[:, 0], y=X_elong[:, 1], mode="markers",
                          marker=dict(color=point_colors, size=5), showlegend=False), row=1, col=2)
for k in range(3):
    ex, ey = cov_ellipse_xy(gmm_full.means_[k], gmm_full.covariances_[k])
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color=comp_colors[k], width=2.5),
                              showlegend=False), row=1, col=2)
fig.update_layout(height=460, width=940, plot_bgcolor="#fafafa", paper_bgcolor="#fafafa",
                   title="Same data, same k=3: a point versus a shape")
fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_yaxes(scaleanchor="x2", scaleratio=1, row=1, col=2)
fig.show()

**Payoff.** k-means scores 0.791 -- respectable, but its round boundaries visibly cut across every elongated cluster's long axis, exactly the villain slide 6 warned about. The full-covariance GMM scores 0.986, and the reason is on screen: its covariance ellipses are not circles, they are the *actual shapes* of the three clouds, so the boundary between components follows the data instead of an assumption about it. The generalisable point: k-means' error here is not "not enough iterations" or "bad luck" -- it is structural, because a single centre plus one number (distance) has no way to represent "long in this direction, narrow in that one," no matter how well it is optimised.

**Idea -- force the freedoms off, one at a time, and watch k-means reappear.** Slide 19's reveal, run for real: refit as `covariance_type="spherical"` (one shared radius, no tilt, no stretch) with equal weights, initialised at k-means' own centres so both methods start from the same place. That machine is still soft -- it still reports fractional responsibilities. Harden it with `.predict()` (an argmax over those responsibilities) and compare the result to k-means' own labels directly, not just to the ground truth.

In [8]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

gmm_spherical = GaussianMixture(n_components=3, covariance_type="spherical",
                                 means_init=km.cluster_centers_, random_state=0).fit(X_elong)
labels_hardened = gmm_spherical.predict(X_elong)  # the hardening step: argmax over responsibilities

# Cluster indices are arbitrary labels, not identities -- align them by best overlap
# before comparing point-by-point (ARI already does this internally; here we do it
# explicitly so we can also show WHICH points disagree).
cm = confusion_matrix(km.labels_, labels_hardened)
row_idx, col_idx = linear_sum_assignment(-cm)
perm = dict(zip(col_idx, row_idx))
labels_aligned = np.array([perm[label] for label in labels_hardened])

agree = (labels_aligned == km.labels_).mean()
ari_vs_kmeans = adjusted_rand_score(labels_hardened, km.labels_)
ari_vs_true = adjusted_rand_score(y_true, labels_hardened)
print(f"identical spherical, equal weight, hardened -- ARI vs ground truth: {ari_vs_true:.3f}")
print(f"agreement with k-means' own labels: {agree:.1%} of points ({(labels_aligned != km.labels_).sum()} disagree), ARI = {ari_vs_kmeans:.3f}")

fig = go.Figure()
disagree_mask = labels_aligned != km.labels_
fig.add_trace(go.Scatter(x=X_elong[~disagree_mask, 0], y=X_elong[~disagree_mask, 1], mode="markers",
                          marker=dict(color=DIM, size=5), name="same label, both methods"))
fig.add_trace(go.Scatter(x=X_elong[disagree_mask, 0], y=X_elong[disagree_mask, 1], mode="markers",
                          marker=dict(color=RED, size=8, symbol="x"), name="disagree"))
fig.update_layout(title="Hardened spherical GMM vs k-means: where do they differ?",
                   width=650, height=500)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

identical spherical, equal weight, hardened -- ARI vs ground truth: 0.603
agreement with k-means' own labels: 92.9% of points (15 disagree), ARI = 0.785


**Payoff.** Switching off shape and weight, then switching off softness, lands on 92.9% of k-means' own partition (15 of 210 points disagree, ARI 0.785 against k-means' labels) -- the handful of disagreements sit right on the boundaries between clusters, exactly where two different random initialisations would be expected to wobble. That is not a coincidence and not an approximation: identical spherical covariance plus equal weights plus a hardened responsibility *is*, term for term, the k-means assignment rule. The takeaway generalises well past this dataset: k-means is not a different algorithm that happens to resemble EM, it is EM with three specific freedoms removed, which is exactly why every failure mode you saw in the k-means lecture -- the boundary lie, the wrong shape -- is really a statement about which freedoms were missing, not a flaw unique to k-means itself.

## Part 3 -- A Model, Not a Map

**Problem.** A partition -- from k-means or from a hardened GMM -- can answer "which group?" and nothing else. Slides 21-22 claimed the *un-hardened* GMM can do two things a partition never could: say how surprising a brand-new point would be, and generate new points that look like it came from the data. Let's cash in both claims on `gmm_full`, the same fitted model from Part 2.

**Ingredients.** Reuse `gmm_full` and `X_elong` from Part 2 -- no new fitting needed, because a density, once fitted, already contains both capabilities.

**Idea -- score, don't just assign.** `score_samples` returns the log-density the model assigns to any point: high for points sitting in the thick of a cluster, low for points the model finds surprising. Score every real data point to see the "normal" range, then score two deliberately unusual points: one sitting far outside all three clusters, and one sitting just outside a cluster's edge -- a genuinely borderline case, not an absurd one.

In [9]:
in_sample_scores = gmm_full.score_samples(X_elong)

far_point = np.array([[14.0, 14.0]])
mild_outlier = np.array([[-3.0, 3.0]])
far_score = gmm_full.score_samples(far_point)[0]
mild_score = gmm_full.score_samples(mild_outlier)[0]

print(f"in-sample log-density: min={in_sample_scores.min():.2f}  mean={in_sample_scores.mean():.2f}  max={in_sample_scores.max():.2f}")
print(f"far-off point (14, 14):        log-density = {far_score:.2f}")
print(f"mild borderline point (-3, 3): log-density = {mild_score:.2f}")

fig = go.Figure()
fig.add_trace(go.Histogram(x=in_sample_scores, marker_color=BLUE, name="real data points", nbinsx=30))
fig.add_vline(x=far_score, line=dict(color=RED, dash="dash"), annotation_text="far point")
fig.add_vline(x=mild_score, line=dict(color=GOLD, dash="dash"), annotation_text="mild outlier")
fig.update_layout(title="How surprising is a point? log-density under the fitted GMM",
                   xaxis_title="log-density", yaxis_title="count of real points", width=700, height=420)
fig.show()

assert far_score < mild_score < in_sample_scores.min()

in-sample log-density: min=-8.56  mean=-3.60  max=-2.50
far-off point (14, 14):        log-density = -215.48
mild borderline point (-3, 3): log-density = -18.34


**Payoff.** Every real data point scores between $-8.56$ and $-2.50$; the mild outlier scores $-18.34$, clearly below that whole band without being absurd; the far-off point scores $-215.48$ -- over 200 natural-log units below the least-typical real point, a factor of more than $e^{200}$ in raw density, not a rounding difference. The generalisable point: `score_samples` turns whatever clustering model you fit into an anomaly detector *for free*, because "how well does this point fit the model" was always a well-defined question the moment the model became a real density -- a partition never had an answer to it at all.

**Idea -- sample, don't just fit.** Because `gmm_full` is a genuine generative story (slide 10: pick a component by weight, draw from that component's Gaussian), it can run in reverse. Draw brand-new points from the fitted model and lay them over the real data: if the model actually learned the shape of the clusters, the synthetic cloud should be visually indistinguishable from the real one.

In [10]:
X_synthetic, component_ids = gmm_full.sample(n_samples=len(X_elong))

fig = go.Figure()
fig.add_trace(go.Scatter(x=X_elong[:, 0], y=X_elong[:, 1], mode="markers",
                          marker=dict(color=DIM, size=5, opacity=0.6), name="real data"))
fig.add_trace(go.Scatter(x=X_synthetic[:, 0], y=X_synthetic[:, 1], mode="markers",
                          marker=dict(color=GOLD, size=5, symbol="circle-open"), name="sampled from the model"))
fig.update_layout(title="Real points vs brand-new points drawn from the fitted density",
                   width=700, height=500)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

**Payoff.** The sampled cloud lands on top of the real one, same three elongated shapes, same relative sizes -- because `.sample()` is running the exact generative story slide 10 described, using the parameters Part 2 fitted, not a new mechanism bolted on afterward. The takeaway: scoring and sampling are not two separate bonus features, they are the same fact -- "this model is a real probability distribution" -- read in two directions, and neither one was available to k-means, whose output is a set of labels with no density behind them at all.

## Challenge -- break it deliberately: the EM singularity

**Problem.** Every fit in Parts 1-3 used a sensible number of well-populated components. Slide 20 warned that unconstrained maximum likelihood tempts you toward more and more components, because likelihood always rises with K. Push that temptation to its logical extreme: will maximum likelihood, run with no safeguards, hold up when a component has almost nothing to fit?

**Ingredients.**

In [11]:
rng = np.random.default_rng(4)
g1 = rng.normal(loc=[0, 0], scale=0.8, size=(6, 2))
g2 = rng.normal(loc=[6, 6], scale=0.8, size=(6, 2))
dup = np.array([[3.0, 3.0], [3.0, 3.0]])  # an exact duplicate pair -- the trap
X_tiny = np.vstack([g1, g2, dup])
print(X_tiny.shape, "points, deliberately squeezed: 4 components asked of only 14 points, two of them identical")

(14, 2) points, deliberately squeezed: 4 components asked of only 14 points, two of them identical


**Idea -- see exactly why an unconstrained fit can cheat.** The 2D Gaussian density involves $1/\sqrt{|\Sigma|}$, where $|\Sigma|$ is the covariance's determinant -- roughly, the area the Gaussian's "spread" covers. If a component's mean lands exactly on a small handful of points and its covariance is allowed to shrink to match them exactly, $|\Sigma|\to 0$ and the density at those points shoots toward infinity. Unconstrained maximum likelihood does not know this is a cheat -- to the raw formula, "assign near-zero spread to two duplicate points" scores as an *arbitrarily good* fit, better than any honest broad Gaussian could ever score. Try it with no floor under the variance at all.

In [12]:
from sklearn.mixture import GaussianMixture

# reg_covar=0.0: nothing stops a component's covariance from collapsing to zero.
try:
    GaussianMixture(n_components=4, covariance_type="full", reg_covar=0.0,
                     random_state=0, n_init=1, max_iter=200).fit(X_tiny)
except ValueError as e:
    print(f"CRASHED outright: {e}\n")

# sklearn's own default (reg_covar=1e-6) is enough to dodge THIS particular crash,
# but not enough to stop the pathology -- watch the average log-likelihood turn positive.
gmm_default = GaussianMixture(n_components=4, covariance_type="full", reg_covar=1e-6,
                               random_state=0, n_init=1, max_iter=200).fit(X_tiny)
dets = [np.linalg.det(c) for c in gmm_default.covariances_]
print(f"avg log-likelihood: {gmm_default.score(X_tiny):.3f}  (positive means the model claims 'more than certain')")
print("det(covariance) per component:", [f"{d:.2e}" for d in dets])
assert gmm_default.score(X_tiny) > 0
assert min(dets) < 1e-5

CRASHED outright: Fitting the mixture model failed because some components have ill-defined empirical covariance (for instance caused by singleton or collapsed samples). Try to decrease the number of components, increase reg_covar, or scale the input data.

avg log-likelihood: 0.301  (positive means the model claims 'more than certain')
det(covariance) per component: ['2.41e-07', '1.26e-01', '1.00e-12', '7.73e-03']


**Payoff.** Two symptoms of the identical disease: with no floor at all, initialisation itself fails outright, because a component landing on the two identical points has a covariance that is undefined, not just small. With sklearn's tiny default floor, the fit survives but one component's covariance determinant collapses to essentially zero, and the average log-likelihood turns *positive* -- a density greater than 1, which for a continuous distribution should sound alarms, not celebration. The generalisable lesson: "the fit that maximises likelihood" and "the fit that describes the data well" are not the same question the moment a component is allowed to shrink onto arbitrarily few points -- unconstrained, the optimum does not exist, only degenerate near-optima arbitrarily far out along the same cheat.

**Idea -- put a floor under variance, then let BIC do its real job.** `reg_covar` adds a small constant to every covariance's diagonal at each M-step, so no component's spread can ever reach exactly zero -- the point-mass exploit is closed by construction, not by luck. With that floor in place, sweep K on real, well-populated data -- the elongated cloud from Part 2, `X_elong`, not this adversarial toy -- and watch BIC's complexity penalty do the job slide 20 promised: raw log-likelihood would keep climbing forever, but BIC turns a real corner.

In [13]:
# The floor: no covariance's diagonal can now fall below reg_covar.
gmm_fixed = GaussianMixture(n_components=4, covariance_type="full", reg_covar=1e-2,
                             random_state=0, n_init=1, max_iter=200).fit(X_tiny)
dets_fixed = [np.linalg.det(c) for c in gmm_fixed.covariances_]
print(f"avg log-likelihood, regularised: {gmm_fixed.score(X_tiny):.3f}  (back to a sane negative number)")
print("det(covariance) per component:", [f"{d:.2e}" for d in dets_fixed])

# Now use the SAME floor to choose K honestly, on data with a real, known answer.
Ks = list(range(1, 8))
bics = [GaussianMixture(n_components=k, covariance_type="full", reg_covar=1e-2,
                         n_init=5, random_state=0).fit(X_elong).bic(X_elong) for k in Ks]

fig = go.Figure(go.Scatter(x=Ks, y=bics, mode="lines+markers", line=dict(color=GOLD)))
best_k = Ks[int(np.argmin(bics))]
fig.add_vline(x=best_k, line=dict(color=RED, dash="dash"), annotation_text=f"BIC's choice: K={best_k}")
fig.update_layout(title="BIC over K, regularised -- the elongated cloud, true K = 3",
                   xaxis_title="K", yaxis_title="BIC (lower is better)", width=650, height=420)
fig.show()

print(f"BIC's choice: K = {best_k}")
assert gmm_fixed.score(X_tiny) < 0
assert best_k == 3

avg log-likelihood, regularised: -1.680  (back to a sane negative number)
det(covariance) per component: ['2.51e-03', '1.36e-01', '1.00e-04', '1.09e-02']


BIC's choice: K = 3


**Payoff.** Regularised, the toy dataset's likelihood drops back to a sane negative number and no component's determinant collapses below the floor `reg_covar` set -- the exploit is gone. On real structured data, BIC now picks $K=3$ exactly, the true number of clusters, with no peeking at `y_true` anywhere in the sweep. The takeaway: `reg_covar` and BIC are doing two different jobs that both have to work for "choose K honestly" to mean anything -- `reg_covar` keeps the *optimisation* well-posed, so no amount of extra components can win by cheating with infinite density, and only once that is true does BIC's complexity penalty measure something real instead of racing against a broken objective it can never catch.

## Reflection

**Problem.** You now own the full arc: the Gaussian as an argued choice of shape, the mixture as a generative story, maximum likelihood as what "fit" means, responsibilities as soft membership, EM as the loop that fits a mixture, k-means as a special case with three freedoms switched off, and density as a bonus that turns a clustering tool into a real model. The point of this last cell isn't a right answer -- it's stating, in your own words, the reasoning each part was built on.

In [14]:
# 1. Part 1 confirmed EM's E-step and M-step are just weighted versions of
#    formulas you already trusted (the 1D Gaussian density, and the
#    sample-mean-and-spread ML fact from slide 12). In your own words, why
#    does treating "how much a point belongs" as a NUMBER between 0 and 1,
#    rather than a yes/no label, let the exact same update rule handle the
#    boundary case k-means could not?
answer_1 = '''
'''

# 2. In Part 2, the full-covariance GMM's ARI beat k-means' by a wide margin
#    on the elongated data, but forcing the GMM to spherical + equal-weight
#    + hardened collapsed its labels onto almost exactly what k-means found
#    on its own. What does that near-agreement tell you about what k-means
#    actually IS, underneath its own algorithm?
answer_2 = '''
'''

# 3. Part 3 let you do two things a partition could never do: score how
#    unlikely a point is, and sample brand-new points. Which of those two
#    capabilities depends on the model being a genuine probability density
#    (one that integrates to 1), and which would still make some sense even
#    for a model that wasn't a proper density?
answer_3 = '''
'''

# 4. The Challenge showed maximum likelihood, left unconstrained, will
#    always prefer a component that collapses onto a handful of points over
#    an honest broad fit. Explain in your own words why "the fit that makes
#    the data look most likely" is not the same question as "the fit that
#    describes the data well" -- and what reg_covar actually buys you
#    against that gap.
answer_4 = '''
'''

# 5. Across this lab, name one moment (Part 1, 2, 3, or the Challenge)
#    where a number computed in code contradicted an intuition you walked
#    in with, and describe what the number showed you instead.
answer_5 = '''
'''


---
*Gaussian Mixture Models: Soft Clustering with EM -- Bluevision-ai Academy*